[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Mixed Column Types &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]


def raw_export(network, seed=7):
    """The network as a station register exports it: some private stations' elevations missing,
    terrain typed by hand in several ways, and a few owners left blank."""
    rng = np.random.default_rng(seed)
    raw = network.copy()
    raw["elevation_m"] = raw["elevation_m"].astype(float)
    hidden = (raw["owner"] == "private").to_numpy() & (rng.uniform(size=len(raw)) < 0.3)
    raw.loc[hidden, "elevation_m"] = np.nan
    retyped = rng.uniform(size=len(raw)) < 0.08
    raw.loc[retyped, "terrain"] = raw.loc[retyped, "terrain"].str.upper() + " "
    blank = rng.uniform(size=len(raw)) < 0.02
    raw.loc[blank, "owner"] = np.nan
    return raw


raw = raw_export(network)
NUMBERS = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]
TEXT = ["terrain", "owner"]

print("scikit-learn", sklearn.__version__, "|", len(raw), "stations in the export")

scikit-learn 1.6.1 | 1200 stations in the export


**1.** Print how many stations with a missing elevation each owner has. Use `raw` and `isna`.

In [2]:
missing = raw[raw["elevation_m"].isna()]

print(missing["owner"].value_counts(dropna=False))

owner
private    67
NaN         1
Name: count, dtype: int64


67 of the 68 are private, as `raw_export` made them. The 68th was private too, and its owner was then left blank, which `dropna=False` counts as `NaN`.

**2.** Give the numbers' imputer `add_indicator=True`, fit the whole pipeline on the training rows, and print the names of the prepared columns. One more column marks the stations whose elevation was filled.

In [3]:
def tidy_text(table):
    return table.assign(terrain=table["terrain"].str.strip().str.lower())


X = raw[NUMBERS + TEXT]
y = raw["jan_mean"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

numbers = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler())
text = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore"))
marked = make_pipeline(
    FunctionTransformer(tidy_text, feature_names_out="one-to-one"),
    ColumnTransformer([("numbers", numbers, NUMBERS), ("text", text, TEXT)]),
    LinearRegression(),
).fit(X_train, y_train)

print(marked[:-1].get_feature_names_out())

['numbers__latitude' 'numbers__longitude' 'numbers__elevation_m'
 'numbers__coast_km' 'numbers__installed'
 'numbers__missingindicator_elevation_m' 'text__terrain_coastal'
 'text__terrain_fjord' 'text__terrain_inland' 'text__terrain_mountain'
 'text__owner_airport' 'text__owner_met' 'text__owner_private'
 'text__owner_road']


The first lines build the split as the worked example did, since this notebook runs on its own. The imputer's `missingindicator_elevation_m` is 1 for a station whose elevation it filled and 0 for the rest, and the scaler after it rescales that column like the others, so the model can learn whether a missing elevation means anything.

**3.** Build the same pipeline with `LogisticRegression()` in place of `LinearRegression()` to predict `snow_on_may_1`, and print its accuracy on a held-out quarter of the stations, with `random_state=0` for the split.

In [4]:
from sklearn.linear_model import LogisticRegression


def tidy_text(table):
    return table.assign(terrain=table["terrain"].str.strip().str.lower())


X = raw[NUMBERS + TEXT]
snow = raw["snow_on_may_1"]
X_train, X_test, snow_train, snow_test = train_test_split(X, snow, test_size=0.25, random_state=0)
numbers = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
text = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore"))
classifier = make_pipeline(
    FunctionTransformer(tidy_text, feature_names_out="one-to-one"),
    ColumnTransformer([("numbers", numbers, NUMBERS), ("text", text, TEXT)]),
    LogisticRegression(),
).fit(X_train, snow_train)

print("accuracy on test rows:", round(classifier.score(X_test, snow_test), 3))

accuracy on test rows: 0.863


Only the last step and the target changed.

**4.** Predict the January mean of a station typed with terrain `Fjord`, owner `road`, latitude 61.2, longitude 7.1, elevation 120 m, 3.5 km from the coast, installed in 1999, with `model`.

In [5]:
def tidy_text(table):
    return table.assign(terrain=table["terrain"].str.strip().str.lower())


X = raw[NUMBERS + TEXT]
y = raw["jan_mean"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

numbers = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
text = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore"))
model = make_pipeline(
    FunctionTransformer(tidy_text, feature_names_out="one-to-one"),
    ColumnTransformer([("numbers", numbers, NUMBERS), ("text", text, TEXT)]),
    LinearRegression(),
).fit(X_train, y_train)

station = pd.DataFrame({
    "latitude": [61.2], "longitude": [7.1], "elevation_m": [120.0], "coast_km": [3.5],
    "installed": [1999], "terrain": ["Fjord"], "owner": ["road"],
})
print(model.predict(station).round(1))

[-1.5]


The tidying step turned `Fjord` into `fjord`, one of the four terrains the encoder learned.

**5.** Build the `ColumnTransformer` with `make_column_selector(dtype_include="number")` and `make_column_selector(dtype_include=object)`, imported from `sklearn.compose`, in place of the two lists of names, and check that `get_feature_names_out` gives the same 13 names.

In [6]:
def tidy_text(table):
    return table.assign(terrain=table["terrain"].str.strip().str.lower())


X = raw[NUMBERS + TEXT]
y = raw["jan_mean"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

from sklearn.compose import make_column_selector

numbers = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
text = make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore"))
by_name = make_pipeline(
    FunctionTransformer(tidy_text, feature_names_out="one-to-one"),
    ColumnTransformer([("numbers", numbers, NUMBERS), ("text", text, TEXT)]),
).fit(X_train)
by_type = make_pipeline(
    FunctionTransformer(tidy_text, feature_names_out="one-to-one"),
    ColumnTransformer([("numbers", numbers, make_column_selector(dtype_include="number")),
                       ("text", text, make_column_selector(dtype_include=object))]),
).fit(X_train)

print(len(by_type.get_feature_names_out()))
print(list(by_type.get_feature_names_out()) == list(by_name.get_feature_names_out()))

13
True


A selector picks the columns by their type when the step is fitted, so it takes every number column and every text column the table has, here the same seven as the two lists.

---

&#8592; **Back to:** [Mixed Column Types](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/12-mixed-column-types.ipynb)  &nbsp;&middot;&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)
